In [3]:
import json
import time
import pandas as pd
from tqdm.notebook import tqdm
from openai import OpenAI

In [4]:
client = OpenAI(
    base_url="http://localhost:11434/v1",
    ##api_key="9d9240b6592a464399b88dafe710da9c.U00kh_iANaeR5HPen36I0rUd"
    api_key="ollama"
)

In [ ]:
INCLUSION = """
    1. The study proposes, evaluates, or applies one or more Large Language Models (LLMs) (e.g., GPT, Llama, Gemma, Mistral, Qwen, Claude, PaLM, DeepSeek, or other transformer-based foundation models) as a primary component of the research.
    2. The study is conducted within an educational context.
    3. The study addresses one or more information extraction (IE) tasks from educational data, such as Named Entity Recognition (NER), Relation Extraction (RE), Attribute extraction, Knowledge extraction, and Structured data generation.
    4. The study is published in a peer-reviewed venue (e.g., journal and conference).
    5. 
    """

EXCLUSION = """
    1. The study did not propose, evaluate, or apply one or more Large Language Models (LLMs) (e.g., GPT, Llama, Gemma, Mistral, Qwen, Claude, PaLM, DeepSeek, or other transformer-based foundation models) as a primary component of the research.
    2. The study is not conducted within an educational context.
    3. The study did not address one or more information extraction (IE) tasks from educational data, such as Named Entity Recognition (NER), Relation Extraction (RE), Attribute extraction, Knowledge extraction, and Structured data generation.
"""

In [6]:
!uv pip install ipywidgets jupyterlab --quiet
from tqdm import tqdm

In [7]:
## pakai batch
BATCH_SIZE = 5
df = pd.read_csv("WOS.csv")
df = df.reset_index(drop=True)
df.head()

,ID,Publication Type,Article Title,Source Title,Conference Title,Abstract
0,1,C,Enhancing Visual Information Extraction with L...,"PATTERN RECOGNITION AND COMPUTER VISION, PRCV ...",7th Chinese Conference on Pattern Recognition ...,"Recently, leveraging large language models (LL..."
1,2,C,FetalExtract-LLM: Structured Information Extra...,"PERINATAL, PRETERM AND PAEDIATRIC IMAGE ANALYS...",10th International Workshop on Perinatal Prete...,Magnetic resonance imaging (MRI) is essential ...
2,3,J,Leveraging Generative AI and Large Language Mo...,KOREAN JOURNAL OF CHEMICAL ENGINEERING,NaN,Process systems engineering (PSE) has long bee...
3,4,J,Use of Machine Learning and Large Language Mod...,ANNUAL REVIEW OF CHEMICAL AND BIOMOLECULAR ENG...,NaN,Rich information in the chemical literature pr...
4,5,J,THE INTEGRATION OF GENERATIVE ARTIFICIAL INTEL...,PRISMA SOCIAL,NaN,The field of artificial intelligence (AI) has ...


In [8]:
def create_batch(df, batch_size):

    batches = []

    for i in range(0, len(df), batch_size):

        batches.append(df.iloc[i:i+batch_size])

    return batches


batches = create_batch(df, BATCH_SIZE)

print(len(batches))

69


In [9]:
from prompt_toolkit import prompt
import json
import re
import time
import pandas as pd

In [10]:
def screening_batch(batch, max_retry=3):

    papers = ""

    for _, row in batch.iterrows():

        papers += f"""
            Paper ID: {row['ID']}

            Abstract:
            {row['Abstract']}

            ----------------------------------------
            """

        prompt = f"""
            You are an expert in Systematic Literature Review.
            Evaluate EVERY paper based on the abstract only.

            Inclusion Criteria:
            {INCLUSION}

            Exclusion Criteria:
            {EXCLUSION}

            Return ONLY a valid JSON array.
            Return exactly this format:

            [
            {{
                "id": 123,
                "decision": "Include",
                "confidence": 92,
                "reason": "Relevant to inclusion criteria."
            }}
            ]

            Papers:
            {papers}
            """

    for attempt in range(max_retry):

        try:

            response = client.chat.completions.create(
                model="gemma3:4b",
                temperature=0,
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )

            content = response.choices[0].message.content.strip()

            content = re.sub(r"^```json", "", content, flags=re.IGNORECASE)
            content = re.sub(r"^```", "", content)
            content = re.sub(r"```$", "", content)
            content = content.strip()

            result = json.loads(content)

            result_df = pd.DataFrame(result)

            # Tambahkan Title & Abstract berdasarkan PaperID
            result_df = result_df.merge(
                batch[["ID", "Article Title", "Abstract"]],
                left_on="id",
                right_on="ID",
                how="left"
            )

            return result_df

        except json.JSONDecodeError:

            print(f"JSON Error (attempt {attempt+1})")

            if attempt == max_retry-1:
                print(content)
                return pd.DataFrame()

        except Exception as e:

            print(f"API Error (attempt {attempt+1}): {e}")

            if attempt == max_retry-1:
                return pd.DataFrame()

            time.sleep(3)

In [11]:
from tqdm.auto import tqdm
import os

os.makedirs("outputs", exist_ok=True)

failed_batches = []

for i, batch in enumerate(
    tqdm(batches, desc="Processing batches"),
    start=1
):

    output_file = f"outputs/output{i}.csv"

    # Skip jika file sudah ada
    if os.path.exists(output_file):
        print(f"Batch {i} already exists, skipping.")
        continue

    try:

        result_df = screening_batch(batch)

        if not result_df.empty:
            result_df.to_csv(output_file, index=False)
            print(f"✓ Saved {output_file}")

    except Exception as e:

        print(f"✗ Batch {i} failed: {e}")

        failed_batches.append({
            "batch": i,
            "error": str(e)
        })

print(f"\nFinished.")
print(f"Failed batches: {len(failed_batches)}")

Processing batches:   0%|          | 0/69 [00:00<?, ?it/s]

✓ Saved outputs/output1.csv
✓ Saved outputs/output2.csv
✓ Saved outputs/output3.csv
✓ Saved outputs/output4.csv
✓ Saved outputs/output5.csv
✓ Saved outputs/output6.csv
✓ Saved outputs/output7.csv
✓ Saved outputs/output8.csv
✓ Saved outputs/output9.csv
✓ Saved outputs/output10.csv
✓ Saved outputs/output11.csv
✓ Saved outputs/output12.csv
✓ Saved outputs/output13.csv
✓ Saved outputs/output14.csv
✓ Saved outputs/output15.csv
✓ Saved outputs/output16.csv
✓ Saved outputs/output17.csv
✓ Saved outputs/output18.csv
✓ Saved outputs/output19.csv
✓ Saved outputs/output20.csv
✓ Saved outputs/output21.csv
✓ Saved outputs/output22.csv
✓ Saved outputs/output23.csv
✓ Saved outputs/output24.csv
✓ Saved outputs/output25.csv
✓ Saved outputs/output26.csv
✓ Saved outputs/output27.csv
✓ Saved outputs/output28.csv
✓ Saved outputs/output29.csv
✓ Saved outputs/output30.csv
✓ Saved outputs/output31.csv
✓ Saved outputs/output32.csv
✓ Saved outputs/output33.csv
✓ Saved outputs/output34.csv
✓ Saved outputs/output3